# A* vs MCTS (Design-based)
This notebook compares A* and MCTS **on Design.py only**.

## 1. Imports & Environment Setup

In [ ]:

try:
    import itertools
    import time
    import os
    import operator
    import numpy as np
    import pandas as pd
    from IPython.display import display

    # Import core classes from graphical_sampling package
    from graphical_sampling.design import Design
    from graphical_sampling.search.astar import AStar
    from graphical_sampling.search.mcts import MCTS

    # Configure pandas display formatting (comma separator and 2 decimal places)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.float_format', lambda x: '{:,.2f}'.format(x))
    print("Imported Successfully")
except KeyError as e:
    print(e)

Imported Successfully


## 2. Scenario Configuration & Parameters (18 MU284_filtered.csv Scenarios)

In [2]:
# Primary study target variable and auxiliary subsets
TARGET_VAR = 'S82'
AUX_VARS = ['P85', 'REV84', 'SS82']
FIP_VARS = ['ME84', 'CS82', 'REG']
SAMPLE_SIZES = [0.10, 0.20]  # Sampling fractions: 10% and 20%
MAX_EVALUATIONS = 4000       # Evaluation budget for fair benchmark

# Generate all 18 experimental scenarios
scenarios = list(itertools.product(AUX_VARS, FIP_VARS, SAMPLE_SIZES))

# Container for storing all scenario outputs
simulation_results = []

print("========== CONFIGURATION LOADED ==========")
print(f"Total scenarios to run: {len(scenarios)}")
print(f"Evaluation limit locked at: {MAX_EVALUATIONS} states")

========== CONFIGURATION LOADED ==========
Total scenarios to run: 18
Evaluation limit locked at: 4000 states


## 3. Population Data & Correlation Matrix (MU284)

In [4]:
# Load MU284 dataset (filtered outliers: N=281)
file_path = os.path.join('populations', 'MU284_filtered.csv')
df_pop = pd.read_csv(file_path)

# Select all variables involved in the study
all_selected_vars = [TARGET_VAR] + AUX_VARS + FIP_VARS

# Calculate and display the full correlation matrix
corr_matrix = df_pop[all_selected_vars].corr()

print(f"========== CORRELATION MATRIX (MU284, N={len(df_pop)}) ==========")
display(corr_matrix)

========== CORRELATION MATRIX (MU284, N=281) ==========


,S82,P85,REV84,SS82,ME84,CS82,REG
S82,1.00,0.87,0.80,0.75,0.85,0.62,-0.15
P85,0.87,1.00,0.91,0.65,0.99,0.67,-0.19
REV84,0.80,0.91,1.00,0.60,0.91,0.61,-0.14
SS82,0.75,0.65,0.60,1.00,0.66,0.15,0.02
ME84,0.85,0.99,0.91,0.66,1.00,0.65,-0.19
CS82,0.62,0.67,0.61,0.15,0.65,1.00,-0.45
REG,-0.15,-0.19,-0.14,0.02,-0.19,-0.45,1.00


## 4. Main Simulation Loop: Memory-Bounded GBFS (A*) vs. Adaptive MCTS

In [5]:
print("========== STARTING 18 SCENARIOS ==========")

for i, (aux, fip, sample_size) in enumerate(scenarios, start=1):
    print(f"\n{'='*50}")
    print(f"Scenario {i}/18 | Aux: {aux} | FIP: {fip} | Size: {int(sample_size*100)}%")
    print(f"{'='*50}")

    # ==========================================
    # 1. Load Data & First-Order Inclusion Probabilities
    # ==========================================
    file_path = os.path.join('populations', 'MU284_filtered.csv')
    df = pd.read_csv(file_path)
    
    N = len(df)
    n = int(np.round(N * sample_size))
    
    fip_data = df[fip].values.astype(float)
    aux_data = df[aux].values.astype(float)
    main_data = df[TARGET_VAR].values.astype(float) 
    
    # Calculate correlations with study variable
    corr_aux_main = np.corrcoef(aux_data, main_data)[0, 1]
    corr_fip_main = np.corrcoef(fip_data, main_data)[0, 1]

    # Proportional-to-size inclusion probabilities scaling
    probs = n * (fip_data / fip_data.sum())
    
    while np.any(probs > 1.0 + 1e-9): 
        certain = probs >= 1.0
        probs[certain] = 1.0
        remaining_n = n - np.sum(certain)
        uncertain = ~certain
        probs[uncertain] = remaining_n * (fip_data[uncertain] / np.sum(fip_data[uncertain]))
    
    probs = np.clip(probs, 0.0, 1.0)
    
    # ==========================================
    # 2. Phase 0: Multi-start Greedy Initialization
    # ==========================================
    criteria = operator.attrgetter('nht_variance')
    
    standard_design = Design(inclusion=probs, variable=aux_data, permute=False)
    std_var = criteria(standard_design)
    
    initial_designs_list = [standard_design]
    population_size = 10
    
    for _ in range(population_size - 1):
        current_design = standard_design.copy()
        current_var = std_var
        
        patience = 50 
        for step in range(patience):
            candidate = current_design.copy()
            rand_switch = np.random.uniform(0.5, 1.0)
            candidate.iterate(random_pull=False, switch_coefficient=rand_switch)
            
            cand_var = criteria(candidate)
            if cand_var <= current_var:
                current_design = candidate
                current_var = cand_var
                
        initial_designs_list.append(current_design)
        
    initial_var = min([criteria(d) for d in initial_designs_list])
    print(f"   Initial Population Best VarNHT (Aux): {initial_var:,.2f}")

    # ==========================================
    # 3. Memory-Bounded GBFS (A*) Execution
    # ==========================================
    print("\n   --- Running A* ---")
    astar = AStar(
        initial_designs=initial_designs_list, 
        switch_coefficient=0.5,
        random_pull=False
    )
    
    t0_astar = time.perf_counter()
    astar.run(
        max_iterations=10000,
        num_new_nodes=50,          
        max_open_set_size=2000,    
        num_changes=(1, 5),        
        top_k=10,
        max_evaluations=MAX_EVALUATIONS
    )
    astar_time = time.perf_counter() - t0_astar
    astar_best_var_aux = astar.best_criteria_value
    
    # Evaluate study variable variance on final A* design
    astar_design_for_main = astar.best_design.copy()
    astar_design_for_main.variable = main_data
    astar_best_var_main = criteria(astar_design_for_main)

    astar_evals = getattr(astar, 'evaluation_count', getattr(astar, 'eval_count', getattr(astar, 'evaluations', MAX_EVALUATIONS)))
    
    # Validity verification
    inc_array_astar = np.array(astar.best_design.inclusion)
    astar_inclusion_sum = np.sum(inc_array_astar)
    astar_is_valid = bool(np.isclose(astar_inclusion_sum, n, atol=1e-5) and np.all((inc_array_astar >= 0) & (inc_array_astar <= 1)))
    validity_status_astar = "VALID" if astar_is_valid else f"INVALID (Sum: {astar_inclusion_sum:.2f}, Expected: {n})"
    
    print(f"   ✔ [A* Finished] Best VarNHT: {astar_best_var_aux:,.2f} | Evals: {astar_evals}/{MAX_EVALUATIONS} | Status: {validity_status_astar}")

    # ==========================================
    # 4. Adaptive MCTS Execution
    # ==========================================
    print("\n   --- Running MCTS ---")
    t0_mcts = time.perf_counter()
    
    mcts = MCTS(
        initial_designs=initial_designs_list,
        criteria=criteria,
        switch_coefficient=1.0,  
        base_changes=20,          
        exploration_constant=10 
    )
    
    best_design_mcts, mcts_best_var_aux = mcts.run(
        max_iterations=10000,    
        max_children_per_node=8,
        max_evaluations=MAX_EVALUATIONS
    )
    mcts_time = time.perf_counter() - t0_mcts

    # Evaluate study variable variance on final MCTS design
    mcts_design_for_main = best_design_mcts.copy()
    mcts_design_for_main.variable = main_data
    mcts_best_var_main = criteria(mcts_design_for_main)

    # Validity verification
    inc_array_mcts = np.array(best_design_mcts.inclusion)
    mcts_inclusion_sum = np.sum(inc_array_mcts)
    mcts_is_valid = bool(np.isclose(mcts_inclusion_sum, n, atol=1e-5) and np.all((inc_array_mcts >= 0) & (inc_array_mcts <= 1)))
    validity_status_mcts = "VALID" if mcts_is_valid else f"INVALID (Sum: {mcts_inclusion_sum:.2f}, Expected: {n})"

    print(f"   ✔ [MCTS Finished] Best VarNHT: {mcts_best_var_aux:,.2f} | Status: {validity_status_mcts}")

    # ==========================================
    # 5. Record Metrics (Exact 15 Columns Matching Target Format)
    # ==========================================
    simulation_results.append({
        'Scenario': i,
        'Aux_Var': aux,
        'FIP_Var': fip,
        'Size': f"{int(sample_size * 100)}%",
        'Corr(Main,Aux)': round(corr_aux_main, 2),
        'Corr(Main,FIP)': round(corr_fip_main, 2),
        'AStar_Aux_Var': astar_best_var_aux,
        'AStar_Main_Var': astar_best_var_main,
        'MCTS_Aux_Var': mcts_best_var_aux,
        'MCTS_Main_Var': mcts_best_var_main,
        'AStar_Evals': astar_evals,
        'AStar_Valid': astar_is_valid,
        'MCTS_Valid': mcts_is_valid,
        'AStar_Time(s)': round(astar_time, 2),
        'MCTS_Time(s)': round(mcts_time, 2)
    })

========== STARTING 18 SCENARIOS ==========

Scenario 1/18 | Aux: P85 | FIP: ME84 | Size: 10%
   Initial Population Best VarNHT (Aux): 10,713.67

   --- Running A* ---
[!] Target evaluations (4000/4000) reached. Stopping A*.
   ✔ [A* Finished] Best VarNHT: 9,081.11 | Evals: 4000/4000 | Status: VALID

   --- Running MCTS ---
=== MULTI-START MCTS (Population Size: 10) ===
Best Initial Criteria: 10713.6708984375
=== RUN STARTED (ADAPTIVE POPULATION MODE) ===

>>> ITER 0 | Depth 1 | Val: 0.00
  ★ BEST: 10713.7 -> 10633.7 (Type: high)
  ★ BEST: 10633.7 -> 10612.8 (Type: high)
  ★ BEST: 10612.8 -> 9989.1 (Type: high)
  ★ BEST: 9989.1 -> 9841.2 (Type: low)

  >>> STAGNATION (8 iters) -> TELEPORT #1 to Best (9841.20)
  ★ BEST: 9841.2 -> 9738.6 (Type: high)
  ★ BEST: 9738.6 -> 9706.7 (Type: high)
  ★ BEST: 9706.7 -> 9521.7 (Type: high)
  ★ BEST: 9521.7 -> 9502.8 (Type: low)
  ★ BEST: 9502.8 -> 9434.6 (Type: low)

  >>> STAGNATION (8 iters) -> TELEPORT #2 to Best (9434.55)
  ★ BEST: 9434.6 -> 94

## 5. Relative Efficiency (RE) Calculation & Performance Summary

In [7]:
# ==========================================
# RE Calculation, Comparison Summary & Export
# ==========================================
print("\n========== ALL SCENARIOS COMPLETED ==========")

# Ensure exact column ordering matching the target table
columns_order = [
    'Scenario', 'Aux_Var', 'FIP_Var', 'Size',
    'Corr(Main,Aux)', 'Corr(Main,FIP)',
    'AStar_Aux_Var', 'AStar_Main_Var',
    'MCTS_Aux_Var', 'MCTS_Main_Var',
    'AStar_Evals', 'AStar_Valid', 'MCTS_Valid',
    'AStar_Time(s)', 'MCTS_Time(s)'
]

df_results = pd.DataFrame(simulation_results)[columns_order]

# Calculate Relative Efficiency (RE) based on Equation (4.1) of the paper
# RE > 100 indicates MCTS performs better (lower variance) than A* (GBFS)
df_results['RE_Aux'] = 100 * (df_results['AStar_Aux_Var'] / df_results['MCTS_Aux_Var'])
df_results['RE_Main'] = 100 * (df_results['AStar_Main_Var'] / df_results['MCTS_Main_Var'])

# Save results to CSV file
output_filename = f'Astar_vs_MCTS_Scenarios_{MAX_EVALUATIONS}evals.csv'
df_results.to_csv(output_filename, index=False)
print(f"Results successfully saved to: {output_filename}")

# Display final scenario table
try:
    display(df_results.style.hide(axis="index").format({
        'Corr(Main,Aux)': '{:.2f}',
        'Corr(Main,FIP)': '{:.2f}',
        'AStar_Aux_Var': '{:,.2f}',
        'AStar_Main_Var': '{:,.2f}',
        'MCTS_Aux_Var': '{:,.2f}',
        'MCTS_Main_Var': '{:,.2f}',
        'AStar_Time(s)': '{:.2f}',
        'MCTS_Time(s)': '{:.2f}'
    }))
except AttributeError:
    display(df_results)

# ==========================================
# Summary Comparison: MCTS Wins vs A* Wins (Auxiliary vs Study Variable)
# ==========================================
mcts_wins_aux = (df_results['RE_Aux'] > 100).sum()
astar_wins_aux = len(df_results) - mcts_wins_aux

mcts_wins_main = (df_results['RE_Main'] > 100).sum()
astar_wins_main = len(df_results) - mcts_wins_main

summary_comparison_data = {
    'Variable Type': ['Optimization Variable (Aux)', f'Study Variable ({TARGET_VAR})'],
    'MCTS Wins (>100 RE)': [f"{mcts_wins_aux}/{len(df_results)}", f"{mcts_wins_main}/{len(df_results)}"],
    'A* (GBFS) Wins': [f"{astar_wins_aux}/{len(df_results)}", f"{astar_wins_main}/{len(df_results)}"],
    'Median RE': [
        f"{df_results['RE_Aux'].median():.2f}%",
        f"{df_results['RE_Main'].median():.2f}%"
    ],
    'RE Range': [
        f"{df_results['RE_Aux'].min():.1f}% - {df_results['RE_Aux'].max():.1f}%",
        f"{df_results['RE_Main'].min():.1f}% - {df_results['RE_Main'].max():.1f}%"
    ]
}

df_summary_comparison = pd.DataFrame(summary_comparison_data)
print("\n" + "="*60)
print("========== ALGORITHM PERFORMANCE COMPARISON SUMMARY ==========")
print("="*60)
display(df_summary_comparison)


========== ALL SCENARIOS COMPLETED ==========
Results successfully saved to: Astar_vs_MCTS_Scenarios_4000evals.csv


Scenario,Aux_Var,FIP_Var,Size,"Corr(Main,Aux)","Corr(Main,FIP)",AStar_Aux_Var,AStar_Main_Var,MCTS_Aux_Var,MCTS_Main_Var,AStar_Evals,AStar_Valid,MCTS_Valid,AStar_Time(s),MCTS_Time(s),RE_Aux,RE_Main
1,P85,ME84,10%,0.87,0.85,"9,081.11","2,817,433.16","9,411.91","2,951,829.44",4000,True,True,35.94,47.32,96.485295,95.447017
2,P85,ME84,20%,0.87,0.85,"3,943.75","1,451,475.42","3,598.62","1,371,693.87",4000,True,True,58.34,57.80,109.590528,105.816279
3,P85,CS82,10%,0.87,0.62,"215,542.44","492,835.19","207,736.78","470,298.70",4000,True,True,44.66,49.34,103.757474,104.791952
4,P85,CS82,20%,0.87,0.62,"82,807.10","238,862.72","73,673.29","236,024.26",4000,True,True,64.73,51.75,112.397726,101.202614
5,P85,REG,10%,0.87,-0.15,"2,338,363.25","1,102,408.30","2,244,425.50","1,066,618.49",4000,True,True,38.07,46.87,104.185381,103.355447
6,P85,REG,20%,0.87,-0.15,"643,188.38","240,089.15","623,762.56","231,977.00",4000,True,True,68.29,54.32,103.114296,103.496960
7,REV84,ME84,10%,0.80,0.85,"4,364,317,696.00","2,920,211.22","4,363,734,528.00","3,083,209.88",4000,True,True,33.91,48.77,100.013364,94.713345
8,REV84,ME84,20%,0.80,0.85,"2,897,534,976.00","1,429,673.50","2,601,698,560.00","1,321,777.63",4000,True,True,55.90,59.59,111.370895,108.162936
9,REV84,CS82,10%,0.80,0.62,"6,701,844,992.00","516,393.10","6,675,375,104.00","526,897.97",4000,True,True,36.03,47.86,100.396530,98.006280
10,REV84,CS82,20%,0.80,0.62,"2,343,730,688.00","240,266.86","2,196,936,960.00","215,198.52",4000,True,True,63.63,51.15,106.681745,111.648933



========== ALGORITHM PERFORMANCE COMPARISON SUMMARY ==========


,Variable Type,MCTS Wins (>100 RE),A* (GBFS) Wins,Median RE,RE Range
0,Optimization Variable (Aux),16/18,2/18,103.73%,96.5% - 112.7%
1,Study Variable (S82),13/18,5/18,103.43%,94.7% - 111.6%
